In [26]:
### conda activate bluemarble

import matplotlib.pyplot as plt
import numpy as np
import os
from netCDF4 import Dataset
import pandas as pd
import cartopy
import cartopy.crs as ccrs
import tqdm
import pickle
import itertools
fpl=10000

def myround(x, prec=2, base=.05):
  return round(base * round(float(x)/base),prec)

#Made by SIT_and_SNOW.ipynb
d = Dataset('inputs/sit_dists_array.nc')
sit_probs = np.array(d['thicknesses'])
sit_dists = np.array(d['sit_dists'])

# Made by Make_Area_partition.ipynb

d = Dataset('inputs/snow_dists_array.nc')
depth_cols = np.array(d['snow_depths'])
snow_dists = np.array(d['snow_dists'])
pen_df = pd.read_csv('inputs/pen_df.csv',index_col='combo')

#Made by SIT_and_SNOW.ipynb
ijs = pd.read_csv('inputs/valid_ijs.csv')
valid_ijs = [(x,y) for x,y in zip(ijs['i'],ijs['j'])]

rf = np.array(Dataset('inputs/rf.nc')['ridged_fraction_filled'])
mpf = np.array(Dataset('inputs/mpf.nc')['melt_pond_fraction_filled'])/100


cum_sit_probs=np.cumsum(sit_probs)

cum_sit_probs

# -np.argmax((1-cum_sit_probs[::-1])>0.03)

def get_pen_from_combo(snow,sit):
    print(snow,sit)

    snow_rnd = myround(snow,prec=2,base=0.01)
    sit_rnd = myround(sit,prec=2,base=0.05)

    combo = (np.round(fpl*(snow_rnd)),
             np.round(fpl*(sit_rnd)))
             
    print(combo)

    pen = float(pen_df.loc[[combo]]['penetration'].iloc[0])

    return pen

level_pens = np.full(shape=(896,608),fill_value=np.nan)
ridged_pens = np.full(shape=(896,608),fill_value=np.nan)

for i,j in tqdm.tqdm(valid_ijs):

    snow_probs = snow_dists[i,j]
    snow_depth_bins = depth_cols
    
    sits = sit_dists[i,j]

    ridged_fraction = rf[i,j]
    melt_pond_fraction = mpf[i,j]

    if np.isnan(sits[0]):        pass
    elif np.isnan(snow_probs[0]):        pass
    elif np.isnan(ridged_fraction):        pass
    elif np.isnan(melt_pond_fraction):        pass
    

    else:
        
        ridged_fraction = rf[i,j]
        melt_pond_fraction = mpf[i,j]

        sit_split = -np.argmax((1-cum_sit_probs[::-1])>0.2)
        
        level_pens_list=[]
        for sit in sits[:sit_split]:
            pens = [get_pen_from_combo(sd,sit) for sd in snow_depth_bins]
            level_pen = np.average(pens,weights=snow_probs)
            level_pens_list.append(level_pen)
        all_level_pen = np.average(level_pens_list,weights=sit_probs[:sit_split])
        level_pens[i,j]=all_level_pen

        if sit_split==0:
            ridged_pens[i,j]=np.nan
        
        ridged_pens_list=[]
        for sit in sits[sit_split:]:
            pens = [get_pen_from_combo(sd,sit) for sd in snow_depth_bins]
            ridged_pen = np.average(pens,weights=snow_probs)
            ridged_pens_list.append(ridged_pen)
        all_ridged_pen = np.average(ridged_pens_list,weights=sit_probs[sit_split:])
        ridged_pens[i,j]=all_ridged_pen
        
# pickle.dump((level_pens,ridged_pens),open('level_ridged_pens.p','wb'))

/tmp/ipykernel_30205/2399089486.py:20: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  sit_probs = np.array(d['thicknesses'])
/tmp/ipykernel_30205/2399089486.py:21: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  sit_dists = np.array(d['sit_dists'])
/tmp/ipykernel_30205/2399089486.py:26: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the m

0.025 0.0
(np.float64(200.0), np.float64(0.0))


KeyError: "None of [Index([(200.0, 0.0)], dtype='object', name='combo')] are in the [index]"

In [23]:
pen_df

,Unnamed: 0,penetration,depth,sit
combo,,,,
"(0, 0)",0,5.000000e-01,0.0,0.00
"(0, 500)",1,4.708823e-01,0.0,0.05
"(0, 1000)",2,4.434602e-01,0.0,0.10
"(0, 1500)",3,4.176351e-01,0.0,0.15
"(0, 2000)",4,3.933139e-01,0.0,0.20
...,...,...,...,...
"(10000, 207500)",42415,6.381381e-18,1.0,20.75
"(10000, 208000)",42416,6.009758e-18,1.0,20.80
"(10000, 208500)",42417,5.659777e-18,1.0,20.85


In [27]:
pen_df.index[0]

(np.float64(200.0), np.float64(0.0))

'(0, 0)'

In [22]:
snow_depth_bins

array(['0.025', '0.075', '0.125', '0.175', '0.225', '0.275', '0.325',
       '0.375', '0.425', '0.475', '0.525', '0.575', '0.625', '0.675',
       '0.725', '0.775', '0.825', '0.875', '0.925', '0.975'], dtype=object)

In [21]:
sits

array([0.        , 0.15573982, 0.31147965, 0.46721947, 0.6229593 ,
       0.77869912, 0.93443895, 1.09017877, 1.2459186 , 1.40165842,
       1.55739825, 1.71313807, 1.8688779 , 2.02461772, 2.18035755])

In [7]:
import matplotlib.pyplot as plt
import numpy as np
import os
from netCDF4 import Dataset
#from regrid import regrid
import pandas as pd
import cartopy
import cartopy.crs as ccrs
import tqdm
import pickle
import itertools

In [19]:
d = Dataset('inputs/snow_dists_array.nc')
d

<class 'netCDF4.Dataset'>
root group (NETCDF4 data model, file format HDF5):
    dimensions(sizes): x(896), y(608), d(20)
    variables(dimensions): float64 latitude(x, y), float64 longitude(x, y), <class 'str'> snow_depths(d), float64 snow_dists(x, y, d)
    groups: 

In [1]:
### conda activate bluemarble

#import h5py
import matplotlib.pyplot as plt
import numpy as np
import os
from netCDF4 import Dataset
#from regrid import regrid
import pandas as pd
import cartopy
import cartopy.crs as ccrs
import tqdm
import pickle
import itertools
fpl=10000

def myround(x, prec=2, base=.05):
  return round(base * round(float(x)/base),prec)

sit_dists,sit_probs = pickle.load(open('sit_dists_array.p','rb'))
snow_dists,depth_cols = pickle.load(open('snow_dists_array.p','rb'))
pen_df=pickle.load(open('pen_df.p','rb'))
valid_ijs = pickle.load(open('valid_ijs.p','rb'))

def get_pen_from_combo(snow,sit):

    snow_rnd = myround(snow,prec=2,base=0.01)
    sit_rnd = myround(sit,prec=2,base=0.05)

    combo = (np.round(fpl*(snow_rnd)),
             np.round(fpl*(sit_rnd)))

    pen = float(pen_df.loc[[combo]]['penetration'].iloc[0])

    return pen

In [8]:
(mpf,rf,rf_args,psn_lon,psn_lat) = pickle.load(open('fractions.p','rb'))

In [9]:
depth_cols

array([np.float64(0.025), np.float64(0.075), np.float64(0.125),
       np.float64(0.175), np.float64(0.225), np.float64(0.275),
       np.float64(0.325), np.float64(0.375), np.float64(0.425),
       np.float64(0.475), np.float64(0.525), np.float64(0.575),
       np.float64(0.625), np.float64(0.675), np.float64(0.725),
       np.float64(0.775), np.float64(0.825), np.float64(0.875),
       np.float64(0.925), np.float64(0.975)], dtype=object)

In [25]:
level_mean_pens = np.full(shape=(896,608),fill_value=np.nan)
ridge_mean_pens = np.full(shape=(896,608),fill_value=np.nan)

for i,j in tqdm.tqdm(valid_ijs):

    snow_probs = snow_dists[i,j]
    snow_depth_bins = depth_cols
    
    sits = sit_dists[i,j]
    sit_probs = sit_probs
    ridged_arg = rf_args[i,j]

    possible_nans = [sits[0],snow_probs[0],ridged_arg]

    if np.isnan(possible_nans).any():
        pass
    else:
        ridged_arg = int(ridged_arg)
        
        
        mean_pens_level=[]
        mean_pens_ridge=[]
        
        for sit in sits[:ridged_arg]:
    
            pens = [get_pen_from_combo(sd,sit) for sd in snow_depth_bins]
    
            mean_pen = np.average(pens,weights=snow_probs)
            mean_pens_level.append(mean_pen)
        
        for sit in sits[ridged_arg:]:
        
            pens = [get_pen_from_combo(sd,sit) for sd in snow_depth_bins]
    
            mean_pen = np.average(pens,weights=snow_probs)
            mean_pens_ridge.append(mean_pen)
        
        
        if mean_pens_level:
            level_mean_pens[i,j] = np.average(mean_pens_level,weights=sit_probs[:ridged_arg])
        if mean_pens_ridge:
            ridge_mean_pens[i,j] = np.average(mean_pens_ridge,weights=sit_probs[ridged_arg:])
        
pickle.dump((level_mean_pens,ridge_mean_pens),open('all_mean_pensv2.p','wb'))

100%|█████████████████████████████████████| 46677/46677 [38:49<00:00, 20.04it/s]


In [23]:
sit_probs

array([0.0646, 0.1415, 0.173 , 0.1272, 0.1114, 0.0824, 0.0665, 0.0541,
       0.0429, 0.0347, 0.0287, 0.024 , 0.0194, 0.016 , 0.0136],
      dtype=float32)

In [22]:
ridged_arg

0

In [19]:
ridged_arg

0